# **Exploracion de movilidad diaria del Lago de Atitlan**

Grupo 3 - CC3084 Data Science.


## **1. Preparacion de datos**

### **1.1. Carga de la fuente**

El archivo `BD_moviidad_diario.xlsx` contiene informacion diaria sobre la presencia de visitantes en distintos destinos turisticos de Guatemala. Para este analisis se carga la hoja `Atitlan`, correspondiente a los municipios del Lago de Atitlan.


In [1]:
from pathlib import Path
import pandas as pd

raiz_proyecto = Path.cwd()
if not (raiz_proyecto / "data" / "raw").is_dir():
    raiz_proyecto = raiz_proyecto.parent

ruta_datos = raiz_proyecto / "data" / "raw" / "BD_moviidad_diario.xlsx"
datos = pd.read_excel(
    ruta_datos,
    sheet_name="Atitlan",
    dtype=str,
    keep_default_na=False,
)


### **1.2. Primeras cinco filas**

Se muestran las primeras cinco filas para reconocer como se organiza cada registro diario y que variables contiene.


In [2]:
datos.head()


,destino,sub_grupo,anio,mes,idfecha,dia_semana,pais,Week,evento,cantidad_visitantes
0,PANAJACHEL,LAGO ATITLAN,2025,4,2025-04-11 00:00:00,Viernes,ALEMANIA,15,Semana Santa,9
1,PANAJACHEL,LAGO ATITLAN,2025,4,2025-04-11 00:00:00,Viernes,ARGENTINA,15,Semana Santa,1
2,PANAJACHEL,LAGO ATITLAN,2025,4,2025-04-11 00:00:00,Viernes,AUSTRALIA,15,Semana Santa,4
3,PANAJACHEL,LAGO ATITLAN,2025,4,2025-04-11 00:00:00,Viernes,AUSTRIA,15,Semana Santa,18
4,PANAJACHEL,LAGO ATITLAN,2025,4,2025-04-11 00:00:00,Viernes,BELGICA,15,Semana Santa,4


## **2. Descripcion de las variables**

Las siguientes descripciones son una propuesta para revisar juntos antes de evaluar la calidad de los datos o acordar su preprocesamiento.

- **destino:** municipio del Lago de Atitlan al que corresponde el registro de visitantes; variable cualitativa nominal.

- **sub_grupo:** zona turistica a la que pertenece el lugar del registro; variable cualitativa nominal. En todas las filas de esta hoja tiene el valor `LAGO ATITLAN`, por lo que no permite diferenciar los registros de este analisis.

- **anio:** anio al que corresponde el registro de visitantes; variable temporal de tipo cuantitativa discreta.

- **mes:** numero del mes al que corresponde el registro; variable cualitativa ordinal de caracter ciclico.

- **idfecha:** fecha del registro, que incluye anio, mes, dia y hora; variable temporal de fecha y hora. En esta hoja todas las horas son `00:00:00`. Una misma fecha puede aparecer en varios registros de municipios y paises diferentes.

- **dia_semana:** dia de la semana al que corresponde la fecha, por ejemplo, lunes o viernes; variable cualitativa ordinal de caracter ciclico.

- **pais:** pais de residencia de los visitantes incluidos en el registro; variable cualitativa nominal.

- **Week:** numero de la semana del calendario a la que corresponde la fecha; variable cualitativa ordinal de caracter ciclico. Los valores de esta hoja coinciden con la numeracion de semanas ISO. El numero se repite entre anios y no identifica por si solo una semana unica. Si se decide conservar esta columna durante el preprocesamiento, se renombrara como `num_semana` para seguir la convencion del proyecto.

- **evento:** etiqueta del evento o feriado asociado a la fecha; variable cualitativa nominal. Cuando la fuente no asigna un evento, utiliza la categoria `Regular`. La etiqueta registrada no demuestra por si sola que el evento se celebre en cada municipio.

- **cantidad_visitantes:** cantidad de visitantes presentes en el municipio y fecha del registro, para el pais de procedencia indicado. Aunque conceptualmente representa un conteo de personas, la fuente incluye valores con decimales, por lo que en su representacion actual se trataria como cuantitativa continua. Una persona puede aparecer en varios dias; sumar esta columna a lo largo del mes no produce visitantes unicos mensuales.


## **3. Revision de calidad de los datos**

Se revisa la hoja completa para identificar faltantes, duplicados, formatos inconsistentes y limites de cobertura. Las conversiones y normalizaciones siguientes se realizan en vistas de diagnostico. El DataFrame `datos` y el archivo original permanecen intactos; todavia no se aplican decisiones de limpieza ni se eliminan columnas.

### **3.1. Estructura, categorias y lectura del archivo**

Se comprueban el tamano y las categorias de la hoja. Los tipos de pandas aparecen como texto porque asi se cargo la fuente; la clasificacion conceptual de cada variable esta documentada en la seccion anterior.

In [3]:
import math
import unicodedata
from openpyxl import load_workbook

datos_originales = datos.copy(deep=True)
marcadores_nulo = {"null", "nan", "none", "n/a", "na"}
texto = datos.astype("string")
vacios = texto.isna() | texto.apply(lambda columna: columna.str.strip().eq(""))
marcadores = texto.apply(lambda columna: columna.str.strip().str.lower().isin(marcadores_nulo))
faltantes = vacios | marcadores

def ejemplos(mascara, columnas, limite=10):
    resultado = datos.loc[mascara.fillna(False), columnas].copy()
    resultado.insert(0, "fila_excel", resultado.index + 2)
    return resultado.head(limite)

estructura = pd.DataFrame({
    "variable": datos.columns,
    "tipo_pandas": [str(datos[columna].dtype) for columna in datos],
    "valores_distintos_originales": [datos[columna].nunique(dropna=False) for columna in datos],
})
estructura["constante"] = estructura["valores_distintos_originales"].eq(1)
print(f"Se revisan {len(datos):,} filas y {len(datos.columns)} columnas.")
estructura

Se revisan 82,952 filas y 10 columnas.


,variable,tipo_pandas,valores_distintos_originales,constante
0,destino,object,10,False
1,sub_grupo,object,1,True
2,anio,object,2,False
3,mes,object,10,False
4,idfecha,object,324,False
5,dia_semana,object,7,False
6,pais,object,105,False
7,Week,object,45,False
8,evento,object,10,False
9,cantidad_visitantes,object,7369,False


In [4]:
categorias = pd.DataFrame([
    {"variable": columna, "valor": valor, "registros": cantidad}
    for columna in ["destino", "sub_grupo", "anio", "mes", "dia_semana", "evento"]
    for valor, cantidad in datos[columna].value_counts(dropna=False).items()
])
categorias

,variable,valor,registros
0,destino,PANAJACHEL,12293
1,destino,SAN PEDRO LA LAGUNA,12172
2,destino,SANTIAGO ATITLAN,9570
3,destino,SAN JUAN LA LAGUNA,9053
4,destino,SAN MARCOS LA LAGUNA,8795
5,destino,SANTA CATARINA PALOPO,7128
6,destino,SAN PABLO LA LAGUNA,7126
7,destino,SAN ANTONIO PALOPO,6643
8,destino,SAN LUCAS TOLIMAN,5899
9,destino,SANTA CRUZ LA LAGUNA,4273


In [5]:
libro = load_workbook(ruta_datos, read_only=True, data_only=False)
try:
    hoja = libro["Atitlan"]
    dimension_declarada = hoja.calculate_dimension()
    hoja.reset_dimensions()
    filas_fisicas, ancho_real = 0, 0
    errores_excel, formulas_excel = [], []
    for fila in hoja.iter_rows():
        filas_fisicas += 1
        ancho_real = max(ancho_real, len(fila))
        for celda in fila:
            if celda.data_type == "e":
                errores_excel.append((celda.coordinate, celda.value))
            elif celda.data_type == "f":
                formulas_excel.append((celda.coordinate, celda.value))
finally:
    libro.close()

revision_excel = pd.DataFrame([
    {"revision": "Dimension declarada por Excel", "resultado": dimension_declarada},
    {"revision": "Filas reales incluyendo cabecera", "resultado": filas_fisicas},
    {"revision": "Columnas reales", "resultado": ancho_real},
    {"revision": "Lectura de todas las filas", "resultado": len(datos) == filas_fisicas - 1},
    {"revision": "Celdas con error de Excel", "resultado": len(errores_excel)},
    {"revision": "Celdas con formulas de Excel", "resultado": len(formulas_excel)},
])
revision_excel

,revision,resultado
0,Dimension declarada por Excel,A1:A1
1,Filas reales incluyendo cabecera,82953
2,Columnas reales,10
3,Lectura de todas las filas,True
4,Celdas con error de Excel,0
5,Celdas con formulas de Excel,0


La dimension declarada del archivo indica una sola celda, aunque la hoja tiene miles de registros. La lectura utilizada recupera el contenido completo. Por eso la dimension declarada no debe utilizarse como unica referencia para leer o validar este libro.

### **3.2. Valores faltantes y escritura de los paises**

Se cuentan por separado los nulos reconocidos por pandas, los textos vacios y los marcadores escritos como `NULL`. Despues se buscan espacios y posibles problemas de codificacion. Los porcentajes corresponden a filas del dataset, no a porcentajes de visitantes.

In [6]:
resumen_faltantes = pd.DataFrame({
    "nulos_pandas": datos.isna().sum(),
    "vacios_texto": texto.apply(lambda columna: columna.str.strip().eq("").fillna(False).sum()),
    "marcadores_texto": marcadores.sum(),
    "total_sin_dato": faltantes.sum(),
})
resumen_faltantes["porcentaje_registros"] = (resumen_faltantes["total_sin_dato"] / len(datos) * 100).round(3)
resumen_faltantes

,nulos_pandas,vacios_texto,marcadores_texto,total_sin_dato,porcentaje_registros
destino,0,0,0,0,0.0
sub_grupo,0,0,0,0,0.0
anio,0,0,0,0,0.0
mes,0,0,0,0,0.0
idfecha,0,0,0,0,0.0
dia_semana,0,0,0,0,0.0
pais,0,28,13,41,0.049
Week,0,0,0,0,0.0
evento,0,0,0,0,0.0
cantidad_visitantes,0,0,0,0,0.0


In [7]:
columnas_contexto = ["destino", "idfecha", "pais", "dia_semana", "Week", "evento", "cantidad_visitantes"]
casos_pais_faltante = ejemplos(faltantes["pais"], columnas_contexto, limite=len(datos))
casos_pais_faltante

,fila_excel,destino,idfecha,pais,dia_semana,Week,evento,cantidad_visitantes
3485,3487,PANAJACHEL,2025-11-26 00:00:00,NULL,Miercoles,48,Regular,2
3527,3529,PANAJACHEL,2025-11-27 00:00:00,NULL,Jueves,48,Regular,2
5541,5543,PANAJACHEL,2026-03-15 00:00:00,NULL,Domingo,11,Regular,1
11020,11022,PANAJACHEL,2026-08-02 00:00:00,,Domingo,31,Regular,2
11133,11135,PANAJACHEL,2026-08-05 00:00:00,,Miercoles,32,Regular,2
11172,11174,PANAJACHEL,2026-08-06 00:00:00,,Jueves,32,Regular,2
11327,11329,PANAJACHEL,2026-08-10 00:00:00,,Lunes,33,Fiestas Agostinas,2
11367,11369,PANAJACHEL,2026-08-11 00:00:00,,Martes,33,Fiestas Agostinas,2
13894,13896,SAN ANTONIO PALOPO,2025-11-26 00:00:00,NULL,Miercoles,48,Regular,1
21489,21491,SAN JUAN LA LAGUNA,2025-11-26 00:00:00,NULL,Miercoles,48,Regular,1


In [8]:
codificacion_danada = texto.apply(lambda columna: columna.str.contains(r"[\u00c3\u00c2\ufffd]", regex=True, na=False))
revision_texto = pd.DataFrame({
    "espacios_exteriores": texto.apply(lambda columna: columna.ne(columna.str.strip()).fillna(False).sum()),
    "posible_codificacion_danada": codificacion_danada.sum(),
})
revision_texto

,espacios_exteriores,posible_codificacion_danada
destino,0,0
sub_grupo,0,0
anio,0,0
mes,0,0
idfecha,0,0
dia_semana,0,0
pais,0,1376
Week,0,0
evento,0,0
cantidad_visitantes,0,0


In [9]:
escrituras_espana = datos.loc[datos["pais"].str.startswith("ESPA"), "pais"].value_counts().rename("registros").to_frame()
escrituras_espana

,registros
pais,
ESPAÃ±A,1376
ESPAÑA,386


Se encontraron dos escrituras de Espana: una correcta y otra con la codificacion danada. La variante debe quedar visible antes de acordar su reparacion. No se ha validado la lista completa de paises contra un catalogo externo.

### **3.3. Duplicados y clave del registro**

La clave esperada es municipio, fecha y pais. Tener la misma fecha en distintos municipios o paises no implica duplicacion. Se comprueba tambien si unificar marcadores de ausencia y la escritura de Espana produciria claves repetidas. Esta normalizacion se aplica solamente a una copia de las claves.

In [10]:
clave = ["destino", "idfecha", "pais"]

def normalizar_etiqueta(valor):
    if pd.isna(valor):
        return ""
    valor = str(valor).replace("ESPAÃ±A", "ESPAÑA").strip().casefold()
    if valor in marcadores_nulo:
        return ""
    return "".join(caracter for caracter in unicodedata.normalize("NFKD", valor) if not unicodedata.combining(caracter))

claves_normalizadas = datos[clave].apply(lambda columna: columna.map(normalizar_etiqueta))
duplicados_exactos = datos.duplicated(keep=False)
duplicados_clave = datos.duplicated(clave, keep=False)
duplicados_normalizados = claves_normalizadas.duplicated(keep=False)
revision_duplicados = pd.DataFrame([
    {"revision": "Filas involucradas en duplicados exactos", "registros": int(duplicados_exactos.sum())},
    {"revision": "Filas involucradas en claves repetidas", "registros": int(duplicados_clave.sum())},
    {"revision": "Filas involucradas en claves normalizadas repetidas", "registros": int(duplicados_normalizados.sum())},
    {"revision": "Filas completamente vacias", "registros": int(vacios.all(axis=1).sum())},
])
revision_duplicados

,revision,registros
0,Filas involucradas en duplicados exactos,0
1,Filas involucradas en claves repetidas,0
2,Filas involucradas en claves normalizadas repe...,0
3,Filas completamente vacias,0


### **3.4. Cantidades, rangos y decimales**

Se comprueba que anio, mes, semana y cantidad de visitantes puedan interpretarse numericamente. Se buscan negativos, valores no finitos y partes decimales. Las cantidades no se redondean ni se eliminan por su magnitud.

In [11]:
columnas_numericas = ["anio", "mes", "Week", "cantidad_visitantes"]
numericos = datos[columnas_numericas].apply(pd.to_numeric, errors="coerce")
revision_numerica = []
for columna in columnas_numericas:
    serie = numericos[columna]
    no_finitos = serie.notna() & ~serie.map(lambda valor: math.isfinite(valor) if pd.notna(valor) else True)
    con_fraccion = serie.notna() & serie.sub(serie.round()).abs().gt(1e-8)
    revision_numerica.append({
        "variable": columna,
        "texto_no_convertible": int((~faltantes[columna] & serie.isna()).sum()),
        "no_finitos": int(no_finitos.sum()),
        "negativos": int(serie.lt(0).sum()),
        "ceros": int(serie.eq(0).sum()),
        "registros_con_decimales": int(con_fraccion.sum()),
        "minimo": serie.min(), "maximo": serie.max(),
    })
revision_numerica = pd.DataFrame(revision_numerica)
revision_numerica

,variable,texto_no_convertible,no_finitos,negativos,ceros,registros_con_decimales,minimo,maximo
0,anio,0,0,0,0,0,2025.0,2026.000000
1,mes,0,0,0,0,0,3.0,12.000000
2,Week,0,0,0,0,0,1.0,52.000000
3,cantidad_visitantes,0,0,0,100,13881,0.0,89817.256248


In [12]:
visitantes = numericos["cantidad_visitantes"]
con_decimales = visitantes.sub(visitantes.round()).abs().gt(1e-8)
decimales_por_pais = datos.loc[con_decimales].groupby("pais").size().rename("registros_con_decimales").to_frame()
decimales_por_pais

,registros_con_decimales
pais,
COSTA RICA,3153
EL SALVADOR,2870
GUATEMALA,3240
HONDURAS,2472
NICARAGUA,2146


In [13]:
ejemplos_decimales = ejemplos(con_decimales, columnas_contexto, limite=5)
ejemplos_decimales

,fila_excel,destino,idfecha,pais,dia_semana,Week,evento,cantidad_visitantes
8,10,PANAJACHEL,2025-04-11 00:00:00,COSTA RICA,Viernes,15,Semana Santa,200.33389747125045
10,12,PANAJACHEL,2025-04-11 00:00:00,EL SALVADOR,Viernes,15,Semana Santa,34.246574847131875
14,16,PANAJACHEL,2025-04-11 00:00:00,GUATEMALA,Viernes,15,Semana Santa,20254.375887061808
15,17,PANAJACHEL,2025-04-11 00:00:00,HONDURAS,Viernes,15,Semana Santa,13.605441912071555
21,23,PANAJACHEL,2025-04-11 00:00:00,NICARAGUA,Viernes,15,Semana Santa,27.88844764562829


### **3.5. Coherencia del calendario y numero de semana**

Se comprueba que la fecha coincida con anio, mes y dia de la semana. Para `Week` se contrasta la semana ISO calculada desde la fecha. El anio ISO de una semana puede ser diferente del anio calendario en los dias cercanos al cambio de anio; esa diferencia no es un error.

In [14]:
fechas = pd.to_datetime(datos["idfecha"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
calendario_iso = fechas.dt.isocalendar()
nombres_dias = {0: "lunes", 1: "martes", 2: "miercoles", 3: "jueves", 4: "viernes", 5: "sabado", 6: "domingo"}
dia_esperado = fechas.dt.dayofweek.map(nombres_dias)
dia_registrado = datos["dia_semana"].map(normalizar_etiqueta)
revision_fechas = pd.DataFrame([
    {"revision": "Fecha no interpretable y no vacia", "registros": int((~faltantes["idfecha"] & fechas.isna()).sum())},
    {"revision": "Anio no entero", "registros": int(numericos["anio"].sub(numericos["anio"].round()).abs().gt(1e-8).sum())},
    {"revision": "Mes no entero", "registros": int(numericos["mes"].sub(numericos["mes"].round()).abs().gt(1e-8).sum())},
    {"revision": "Semana no entera", "registros": int(numericos["Week"].sub(numericos["Week"].round()).abs().gt(1e-8).sum())},
    {"revision": "Mes fuera de 1 a 12", "registros": int((numericos["mes"].notna() & ~numericos["mes"].between(1, 12)).sum())},
    {"revision": "Semana fuera de 1 a 53", "registros": int((numericos["Week"].notna() & ~numericos["Week"].between(1, 53)).sum())},
    {"revision": "Anio no coincide con fecha", "registros": int((fechas.notna() & numericos["anio"].notna() & fechas.dt.year.ne(numericos["anio"])).sum())},
    {"revision": "Mes no coincide con fecha", "registros": int((fechas.notna() & numericos["mes"].notna() & fechas.dt.month.ne(numericos["mes"])).sum())},
    {"revision": "Dia de semana no coincide con fecha", "registros": int((fechas.notna() & dia_registrado.ne(dia_esperado)).sum())},
    {"revision": "Week no coincide con semana ISO", "registros": int((fechas.notna() & numericos["Week"].notna() & numericos["Week"].ne(calendario_iso.week.astype(float))).sum())},
    {"revision": "Hora distinta de medianoche", "registros": int((fechas.notna() & fechas.ne(fechas.dt.normalize())).sum())},
])
revision_fechas

,revision,registros
0,Fecha no interpretable y no vacia,0
1,Anio no entero,0
2,Mes no entero,0
3,Semana no entera,0
4,Mes fuera de 1 a 12,0
5,Semana fuera de 1 a 53,0
6,Anio no coincide con fecha,0
7,Mes no coincide con fecha,0
8,Dia de semana no coincide con fecha,0
9,Week no coincide con semana ISO,0


In [15]:
cambio_anio_iso = fechas.notna() & calendario_iso.year.astype(float).ne(fechas.dt.year)
casos_semana = pd.DataFrame({
    "fecha": fechas[cambio_anio_iso],
    "anio_calendario": numericos.loc[cambio_anio_iso, "anio"],
    "semana_registrada": numericos.loc[cambio_anio_iso, "Week"],
    "anio_iso": calendario_iso.loc[cambio_anio_iso, "year"],
}).drop_duplicates().sort_values("fecha")
print("Registros en dias cuyo anio ISO difiere del calendario:", int(cambio_anio_iso.sum()))
casos_semana

Registros en dias cuyo anio ISO difiere del calendario: 960


,fecha,anio_calendario,semana_registrada,anio_iso
4749,2025-12-29,2025,1,2026
4797,2025-12-30,2025,1,2026
4848,2025-12-31,2025,1,2026


Si se conserva o se recalcula el numero de semana, el renombre propuesto es `num_semana`. Para un analisis semanal, convendra identificar tambien el anio ISO o la fecha de inicio de la semana: combinar sin mas el anio calendario con la semana ISO puede separar una misma semana en dos grupos. Esta observacion no modifica las columnas actuales.

### **3.6. Cobertura diaria y meses parciales**

Se revisan los dias disponibles por mes y por municipio. La cobertura se mide con fechas distintas, no sumando visitantes. Los meses con ventanas de eventos no se presentan como si tuvieran datos de todos sus dias.

In [16]:
periodos = pd.period_range(fechas.min().to_period("M"), fechas.max().to_period("M"), freq="M")
cobertura_mensual = (
    pd.DataFrame({"fecha": fechas, "periodo": fechas.dt.to_period("M")})
    .groupby("periodo")["fecha"].agg(dias_observados="nunique", inicio="min", fin="max")
    .reindex(periodos)
)
cobertura_mensual.index.name = "periodo"
cobertura_mensual["dias_observados"] = cobertura_mensual["dias_observados"].fillna(0).astype(int)
cobertura_mensual["dias_del_mes"] = cobertura_mensual.index.days_in_month
cobertura_mensual["estado"] = "Parcial"
cobertura_mensual.loc[cobertura_mensual.dias_observados.eq(0), "estado"] = "Sin registros"
cobertura_mensual.loc[cobertura_mensual.dias_observados.eq(cobertura_mensual.dias_del_mes), "estado"] = "Completo"
print("Cero dias observados significa ausencia de filas, no cero visitantes.")
cobertura_mensual

Cero dias observados significa ausencia de filas, no cero visitantes.


,dias_observados,inicio,fin,dias_del_mes,estado
periodo,,,,,
2025-04,10,2025-04-11,2025-04-20,30,Parcial
2025-05,0,NaT,NaT,31,Sin registros
2025-06,4,2025-06-27,2025-06-30,30,Parcial
2025-07,0,NaT,NaT,31,Sin registros
2025-08,0,NaT,NaT,31,Sin registros
2025-09,30,2025-09-01,2025-09-30,30,Completo
2025-10,31,2025-10-01,2025-10-31,31,Completo
2025-11,30,2025-11-01,2025-11-30,30,Completo
2025-12,31,2025-12-01,2025-12-31,31,Completo


In [17]:
calendario_municipal = (
    pd.DataFrame({"destino": datos.destino, "fecha": fechas, "periodo": fechas.dt.to_period("M")})
    .groupby(["destino", "periodo"])["fecha"].nunique().unstack()
    .reindex(columns=periodos)
)
calendario_municipal = calendario_municipal.fillna(0).astype(int)
municipios_por_fecha = datos.groupby("idfecha")["destino"].nunique()
print("Municipios distintos:", datos.destino.nunique())
print("Fechas distintas:", fechas.nunique())
print("Fechas presentes en todos los municipios:", int(municipios_por_fecha.eq(datos.destino.nunique()).sum()))
print("La tabla cuenta dias observados, no visitantes.")
calendario_municipal

Municipios distintos: 10
Fechas distintas: 324
Fechas presentes en todos los municipios: 324
La tabla cuenta dias observados, no visitantes.


periodo,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
destino,,,,,,,,,,,,,,,,,,
PANAJACHEL,10,0,4,0,0,30,31,30,31,0,0,31,30,31,30,31,31,4
SAN ANTONIO PALOPO,10,0,4,0,0,30,31,30,31,0,0,31,30,31,30,31,31,4
SAN JUAN LA LAGUNA,10,0,4,0,0,30,31,30,31,0,0,31,30,31,30,31,31,4
SAN LUCAS TOLIMAN,10,0,4,0,0,30,31,30,31,0,0,31,30,31,30,31,31,4
SAN MARCOS LA LAGUNA,10,0,4,0,0,30,31,30,31,0,0,31,30,31,30,31,31,4
SAN PABLO LA LAGUNA,10,0,4,0,0,30,31,30,31,0,0,31,30,31,30,31,31,4
SAN PEDRO LA LAGUNA,10,0,4,0,0,30,31,30,31,0,0,31,30,31,30,31,31,4
SANTA CATARINA PALOPO,10,0,4,0,0,30,31,30,31,0,0,31,30,31,30,31,31,4
SANTA CRUZ LA LAGUNA,10,0,4,0,0,30,31,30,31,0,0,31,30,31,30,31,31,4


La tabla diferencia los diez meses completos de las ventanas parciales de abril y junio de 2025 y septiembre de 2026. Enero y febrero de 2026 no tienen registros. Tampoco se supone que una combinacion pais-fecha ausente equivalga a cero visitantes: no existe un contrato que garantice todas las combinaciones posibles. La base diaria no se proyectara como parte del pronostico mensual solicitado por el proyecto.

### **3.7. Etiquetas y ventanas de eventos**

Se comprueba si una misma fecha tiene etiquetas distintas entre paises o municipios. Despues se muestran las fechas efectivamente cubiertas por cada evento. Un intervalo entre inicio y fin no garantiza cobertura continua y la etiqueta no confirma por si sola la celebracion local del evento.

In [18]:
etiquetas_por_fecha = datos.groupby("idfecha")["evento"].nunique()
etiquetas_por_municipio_fecha = datos.groupby(["destino", "idfecha"])["evento"].nunique()
revision_eventos = pd.DataFrame([
    {"revision": "Fechas con mas de una etiqueta entre municipios o paises", "casos": int(etiquetas_por_fecha.gt(1).sum())},
    {"revision": "Municipio-fecha con mas de una etiqueta", "casos": int(etiquetas_por_municipio_fecha.gt(1).sum())},
    {"revision": "Fechas etiquetadas Regular", "casos": int(datos.loc[datos.evento.eq("Regular"), "idfecha"].nunique())},
    {"revision": "Fechas con etiqueta de evento", "casos": int(datos.loc[~datos.evento.eq("Regular"), "idfecha"].nunique())},
])
revision_eventos

,revision,casos
0,Fechas con mas de una etiqueta entre municipio...,0
1,Municipio-fecha con mas de una etiqueta,0
2,Fechas etiquetadas Regular,258
3,Fechas con etiqueta de evento,66


In [19]:
fechas_eventos = pd.DataFrame({"evento": datos.evento, "fecha": fechas}).drop_duplicates()
ventanas_eventos = (
    fechas_eventos.assign(anio=fechas_eventos.fecha.dt.year)
    .groupby(["anio", "evento"])["fecha"]
    .agg(inicio="min", fin="max", dias_observados="nunique")
)
ventanas_eventos

inicio        fin  dias_observados
anio evento                                                       
2025 15 de Septiembre       2025-09-15 2025-09-15                1
     20 de Octubre          2025-10-17 2025-10-20                4
     Año Nuevo              2025-12-30 2025-12-31                2
     Día de los Santos      2025-10-31 2025-10-31                1
     Día del Ejército       2025-06-27 2025-06-30                4
     Festival de las Flores 2025-11-01 2025-11-10               10
     Festival de las Luces  2025-09-13 2025-09-20                7
     Regular                2025-09-01 2025-12-29               97
     Semana Santa           2025-04-11 2025-04-20               10
2026 15 de Septiembre       2026-09-15 2026-09-15                1
     Día del Ejército       2026-06-26 2026-06-29                4
     Festival de las Luces  2026-09-12 2026-09-14                3
     Fiestas Agostinas      2026-08-10 2026-08-18                9
     Regular                2026-03-01 2026-08-31              161
     Semana Santa           2026-03-27 2026-04-05               10

La etiqueta de Ano Nuevo solo tiene el 30 y 31 de diciembre de 2025; no hay una observacion del 1 de enero de 2026. La etiqueta Dia de los Santos figura el 31 de octubre, mientras del 1 al 10 de noviembre aparece Festival de las Flores. Se documentan las ventanas tal como estan registradas, sin corregir fechas o inferir celebraciones a partir del nombre. Para comparar afluencia de eventos con dias regulares, sera necesario definir una referencia compatible con estas coberturas parciales.

### **3.8. Distribucion, ceros y valores grandes**

Se muestran los valores de mayor magnitud y los registros con cero visitantes. Un cero registrado no equivale a una fecha sin cobertura. Tampoco se descartan valores grandes mediante un umbral general, porque los municipios y mercados tienen escalas distintas.

In [20]:
distribucion = visitantes.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).rename("cantidad_visitantes").to_frame()
distribucion

,cantidad_visitantes
count,82952.000000
mean,381.581509
std,2377.943378
min,0.000000
25%,1.000000
50%,4.000000
75%,16.000000
95%,251.000000
99%,12867.088346
max,89817.256248


In [21]:
indices_mayores = visitantes.nlargest(10).index
mayores_registros = datos.loc[indices_mayores, columnas_contexto].copy()
mayores_registros.insert(0, "fila_excel", mayores_registros.index + 2)
mayores_registros

,fila_excel,destino,idfecha,pais,dia_semana,Week,evento,cantidad_visitantes
927,929,PANAJACHEL,2025-09-14 00:00:00,GUATEMALA,Domingo,37,Festival de las Luces,89817.25624751103
50691,50693,SAN PEDRO LA LAGUNA,2025-09-14 00:00:00,GUATEMALA,Domingo,37,Festival de las Luces,73647.92944287574
50266,50268,SAN PEDRO LA LAGUNA,2025-06-29 00:00:00,GUATEMALA,Domingo,26,Día del Ejército,67436.0161609366
476,478,PANAJACHEL,2025-06-29 00:00:00,GUATEMALA,Domingo,26,Día del Ejército,60065.14605336387
4673,4675,PANAJACHEL,2025-12-27 00:00:00,GUATEMALA,Sabado,52,Regular,57600.553030285344
891,893,PANAJACHEL,2025-09-13 00:00:00,GUATEMALA,Sabado,37,Festival de las Luces,56035.160588743965
446,448,PANAJACHEL,2025-06-28 00:00:00,GUATEMALA,Sabado,26,Día del Ejército,55881.80500115101
50234,50236,SAN PEDRO LA LAGUNA,2025-06-28 00:00:00,GUATEMALA,Sabado,26,Día del Ejército,55328.059077656944
4721,4723,PANAJACHEL,2025-12-28 00:00:00,GUATEMALA,Domingo,52,Regular,54810.38978000496
4866,4868,PANAJACHEL,2025-12-31 00:00:00,GUATEMALA,Miercoles,1,Año Nuevo,54566.76761811273


In [22]:
print("Registros con cantidad de visitantes igual a cero:", int(visitantes.eq(0).sum()))
ejemplos_ceros = ejemplos(visitantes.eq(0), columnas_contexto, limite=10)
ejemplos_ceros

Registros con cantidad de visitantes igual a cero: 100


,fila_excel,destino,idfecha,pais,dia_semana,Week,evento,cantidad_visitantes
19,21,PANAJACHEL,2025-04-11 00:00:00,JAMAICA,Viernes,15,Semana Santa,0
56,58,PANAJACHEL,2025-04-12 00:00:00,JAMAICA,Sabado,15,Semana Santa,0
87,89,PANAJACHEL,2025-04-13 00:00:00,HONG KONG,Domingo,15,Semana Santa,0
91,93,PANAJACHEL,2025-04-13 00:00:00,JAMAICA,Domingo,15,Semana Santa,0
128,130,PANAJACHEL,2025-04-14 00:00:00,JAMAICA,Lunes,16,Semana Santa,0
145,147,PANAJACHEL,2025-04-14 00:00:00,UCRANIA,Lunes,16,Semana Santa,0
190,192,PANAJACHEL,2025-04-15 00:00:00,UCRANIA,Martes,16,Semana Santa,0
664,666,PANAJACHEL,2025-09-05 00:00:00,NUEVA ZELANDA,Viernes,36,Regular,0
1238,1240,PANAJACHEL,2025-09-24 00:00:00,REINO UNIDO,Miercoles,39,Regular,0
1256,1258,PANAJACHEL,2025-09-25 00:00:00,HONG KONG,Jueves,39,Regular,0


### **3.9. Resumen de hallazgos para discutir**

Este resumen recoge las observaciones y posibles decisiones para una fase posterior. No se ha aplicado ninguna limpieza, reclasificacion, eliminacion de columnas ni renombre.

In [23]:
resumen_calidad = pd.DataFrame([
    {"hallazgo": "Pais no identificado", "evidencia": f"{int(vacios['pais'].sum())} vacios y {int(marcadores['pais'].sum())} marcadores de texto", "propuesta_para_discutir": "Unificar como nulo; conservar filas y mostrar Pais no identificado al presentar."},
    {"hallazgo": "Codificacion danada de Espana", "evidencia": str(escrituras_espana['registros'].to_dict()), "propuesta_para_discutir": "Unificar la variante danada con la escritura correcta; verificar claves despues."},
    {"hallazgo": "Cantidad con decimales", "evidencia": f"{int(con_decimales.sum())} registros en {len(decimales_por_pais)} paises", "propuesta_para_discutir": "Conservar precision y separar el formato de presentacion."},
    {"hallazgo": "Ceros registrados", "evidencia": f"{int(visitantes.eq(0).sum())} filas", "propuesta_para_discutir": "Conservarlos; no confundirlos con dias o meses sin registros."},
    {"hallazgo": "Cobertura desigual", "evidencia": str(cobertura_mensual.estado.value_counts().to_dict()), "propuesta_para_discutir": "Distinguir meses completos, ventanas de evento y periodos sin dato."},
    {"hallazgo": "Semana y anio ISO", "evidencia": f"{int(cambio_anio_iso.sum())} registros en {len(casos_semana)} fechas del cambio de anio", "propuesta_para_discutir": "Si se conserva Week, renombrar a num_semana; agrupar con anio ISO o inicio de semana."},
    {"hallazgo": "Eventos con ventanas parciales", "evidencia": "Ano Nuevo solo el 30 y 31 de diciembre; etiquetas compartidas entre municipios", "propuesta_para_discutir": "Conservar evidencia de fechas; no asumir celebracion local ni cobertura completa."},
    {"hallazgo": "Columnas constantes o derivables", "evidencia": "sub_grupo es constante; anio, mes, dia_semana y Week se obtienen de idfecha", "propuesta_para_discutir": "Discutir que columnas conservar segun los analisis del dashboard."},
    {"hallazgo": "Dimension Excel incorrecta", "evidencia": f"Declarada {dimension_declarada}; reales {filas_fisicas} filas con cabecera", "propuesta_para_discutir": "Validar que la lectura recupere la hoja completa."},
])
assert datos.equals(datos_originales), "La revision modifico los datos originales."
print("Control final: el DataFrame original permanece intacto.")
resumen_calidad

Control final: el DataFrame original permanece intacto.


,hallazgo,evidencia,propuesta_para_discutir
0,Pais no identificado,28 vacios y 13 marcadores de texto,Unificar como nulo; conservar filas y mostrar ...
1,Codificacion danada de Espana,"{'ESPAÃ±A': 1376, 'ESPAÑA': 386}",Unificar la variante danada con la escritura c...
2,Cantidad con decimales,13881 registros en 5 paises,Conservar precision y separar el formato de pr...
3,Ceros registrados,100 filas,Conservarlos; no confundirlos con dias o meses...
4,Cobertura desigual,"{'Completo': 10, 'Sin registros': 5, 'Parcial'...","Distinguir meses completos, ventanas de evento..."
5,Semana y anio ISO,960 registros en 3 fechas del cambio de anio,"Si se conserva Week, renombrar a num_semana; a..."
6,Eventos con ventanas parciales,Ano Nuevo solo el 30 y 31 de diciembre; etique...,Conservar evidencia de fechas; no asumir celeb...
7,Columnas constantes o derivables,"sub_grupo es constante; anio, mes, dia_semana ...",Discutir que columnas conservar segun los anal...
8,Dimension Excel incorrecta,Declarada A1:A1; reales 82953 filas con cabecera,Validar que la lectura recupere la hoja completa.


## **4. Criterios de preprocesamiento**

A partir de la revision de calidad, se documenta como se preparara la hoja `Atitlan` del archivo `BD_moviidad_diario.xlsx`. Los criterios siguientes orientaran la implementacion posterior del pipeline; en este notebook no se aplican transformaciones ni se modifica el archivo original. Se conservaran nueve columnas y todos los registros; se retirara unicamente la columna constante `sub_grupo`.

### **4.1. Valores faltantes y escritura de los paises**

- En `pais`, se unificaran las 28 celdas vacias y los 13 valores escritos como `NULL` bajo un valor nulo real. Se conservaran los 41 registros, porque siguen aportando informacion para los analisis que no requieren conocer el pais.
- Al presentar los resultados, estos registros se agruparan bajo la etiqueta **Pais no identificado**. Esa etiqueta no se guardara como si fuera un pais real, ni se asignaran los registros automaticamente al grupo nacional o extranjero.
- Se reparara la variante `ESPAÃ±A`, presente en 1,376 registros, para unificarla con `ESPAÑA`, que aparece correctamente en 386 registros. Es una correccion de escritura, no una imputacion ni una estimacion de procedencia.
- Despues de unificar los marcadores de ausencia y la escritura de Espana, se volvera a validar la clave municipio, fecha y pais. La revision de diagnostico no encontro claves repetidas al aplicar esas equivalencias.

### **4.2. Tipos de datos y columnas temporales**

- El dataset procesado conservara `destino`, `anio`, `mes`, `idfecha`, `dia_semana`, `pais`, `num_semana`, `evento` y `cantidad_visitantes`.
- Se convertira `idfecha` a fecha y hora, `anio`, `mes` y `num_semana` a enteros, y `cantidad_visitantes` a un valor numerico con decimales. Las variables categoricas mantendran su significado original.
- Se conservara la precision disponible de las cantidades para realizar los calculos. El formato de presentacion se aplicara al resultado final y no modificara los valores utilizados en los analisis.
- Se retirara `sub_grupo` del dataset procesado porque tiene el mismo valor en todas las filas. El alcance Lago de Atitlan quedara documentado sin repetir esa etiqueta en cada registro.
- Se conservaran anio, mes, dia de la semana y semana ISO, aunque puedan obtenerse de `idfecha`, para facilitar los filtros y las agrupaciones del dashboard. Se validara su correspondencia con la fecha.
- Se renombrara `Week` como `num_semana`, segun la convencion acordada. Para agrupar semanas se utilizara tambien el anio ISO o la fecha de inicio de la semana: los dias del 29 al 31 de diciembre de 2025 pertenecen a la primera semana ISO de 2026. No se corregira ese numero de semana como si fuera un error.

### **4.3. Cantidades, cobertura y eventos**

- Se conservaran los 100 registros con cero visitantes. Un cero presente en la fuente no equivale a un dia o mes sin cobertura. Tampoco se eliminaran valores grandes solo por su magnitud ni se deduplicara sin evidencia.
- Se distinguiran los diez meses completos de las ventanas parciales de abril y junio de 2025 y septiembre de 2026. No se agregaran ceros para completar dias ausentes ni se interpretaran las ventanas de eventos como meses completos.
- Enero y febrero de 2026 permaneceran sin observaciones diarias. El pronostico solicitado por el proyecto corresponde a la base mensual; esta limpieza no proyectara la serie diaria ni los feriados sin cobertura.
- Se conservaran las etiquetas originales de `evento`, incluida `Regular`, junto con sus fechas efectivamente observadas. No se corregira un calendario ni se afirmara que un festival se celebro en cada municipio solo por el nombre de la etiqueta.
- Las comparaciones posteriores entre eventos y dias regulares deberan declarar sus ventanas y utilizar una referencia compatible con la cobertura disponible.
- La cantidad de visitantes representa presencia diaria: una persona puede figurar en varias fechas. Su suma no se presentara como visitantes unicos del mes, ni se sumaran municipios para fabricar un total deduplicado del lago.

### **4.4. Validaciones y ejecucion del pipeline**

- La lectura debera recuperar todas las filas aunque el archivo declare una dimension incorrecta. Antes y despues de las transformaciones, se validaran los nombres de columnas, los diez municipios, la clave del registro y la coherencia entre fecha, anio, mes, dia de semana y semana ISO.
- Se verificara que la reparacion de paises y el tipado no eliminen observaciones ni cambien las cantidades de visitantes. Los valores faltantes de pais se documentaran como informacion no identificada en la fuente, sin imputacion.
- Las transformaciones se implementaran posteriormente en scripts por etapas dentro de `src/`, con rutas y contratos en `config.py` y utilidades compartidas en `utils.py`. Las salidas se guardaran en `data/processed/`, conservando intactos los archivos de `data/raw/`.
- El diccionario del dataset procesado registrara las nueve columnas acordadas, sus nombres, tipos y reglas de limpieza. Este notebook conservara las tablas que respaldan los criterios.
